# Thermal Detective — chip-temperature PINNs, all stages
Reconstructing chip temperature, power hotspots and hidden defects from sparse sensors.

Run **Run All** with *GPU* and *Internet* on. Every cell calls a plain `.py` script; all outputs land in
`/kaggle/working/chip-pinn/results/`. Each training run checkpoints and resumes, so if the session restarts
just run all cells again — finished runs are skipped.

| | approx. time on a T4 (`QUICK = False`) |
|---|---|
| Stages 0–2 (setup, solver, tests) | ~2 min |
| Stage 3 forward PINN | ~4 min |
| Stage 4 inverse k (10 runs) | ~25 min |
| Stage 5 inverse power map (3 layouts + 4 prior variants) | ~35 min |
| Stage 6 defect detection (2 runs) | ~15 min |
| Stage 7 ablation (8 runs) | ~35 min |
| Stage 8 sensor study (8 runs) + parametric | ~45 min |

Total ≈ 3 h. `QUICK = True` runs the whole pipeline with short training (~25 min) to check everything works end-to-end.
Use **Save Version → Quick Save** afterwards to keep the outputs without re-running.

In [ ]:
import os, subprocess, sys
QUICK = False   # True: short training everywhere (end-to-end check); False: full results

PROJECT = "/kaggle/working/chip-pinn" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
REPO_URL = "https://github.com/cosmicwonder338-oss/ireProj2.git"   # repo that contains chip-pinn/
BRANCH = "claude/thermal-detective-pinn-8b23c9"                    # branch with chip-pinn/ (main once merged)
if os.path.isdir("/kaggle/working"):
    if not os.path.isdir("/kaggle/working/_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/kaggle/working/_repo"], check=True)
    else:
        subprocess.run(["git", "-C", "/kaggle/working/_repo", "pull", "--ff-only"], check=True)
    os.makedirs(PROJECT, exist_ok=True)
    subprocess.run("cp -r /kaggle/working/_repo/chip-pinn/. /kaggle/working/chip-pinn/", shell=True, check=True)
os.chdir(PROJECT); sys.path.insert(0, PROJECT)

EXTRA = ("--set training.adam_iters=400 training.lbfgs_iters=100 training.eval_every=200 "
         "training.snapshot_every=50 training.time_budget_min=3 stage4.baseline_draws=3") if QUICK else ""

from IPython.display import Image, Markdown, display
def show(path):
    if os.path.exists(path): display(Image(path))
    else: print("missing:", path)
def show_md(path):
    if os.path.exists(path): display(Markdown(open(path, encoding="utf-8").read()))
    else: print("missing:", path)
print("project:", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU'

## Stage 0 — setup smoke test + all unit tests

In [ ]:
!python scripts/stage0_smoke.py --config configs/stage0_smoke.yaml
!python tests/run_tests.py

## Stage 1 — finite-volume reference solutions (3 layouts × 2 cooling models)

In [ ]:
!python scripts/stage1_reference.py --config configs/stage1_reference.yaml
show("results/stage1_reference/layouts_backside.png")

## Stage 2 — verification: solver convergence + autograd residual

In [ ]:
!python scripts/stage2_verify.py --config configs/stage2_verify.yaml
show("results/stage2_verify/convergence.png")

## Stage 3 — forward PINN (known k and q → T)

In [ ]:
!python scripts/stage3_forward.py --config configs/stage3_forward.yaml $EXTRA
show("results/stage3_forward/comparison.png"); show("results/stage3_forward/training.png")

## Stage 4 — inverse 1: unknown constant k from 15 noisy sensors (noise 0/1/2/5 %)

In [ ]:
!python scripts/stage4_inverse_k.py --config configs/stage4_inverse_k.yaml $EXTRA
show_md("results/stage4_inverse_k/summary.md")
show("results/stage4_inverse_k/k_error_vs_noise.png"); show("results/stage4_inverse_k/k_trajectory.png")

## Stage 5 — inverse 2 (main result): unknown power map from 15 sensors, hotspot location

In [ ]:
!python scripts/stage5_inverse_q.py --config configs/stage5_inverse_q.yaml $EXTRA
show_md("results/stage5_inverse_q/summary.md")
for lay in ["quad_core", "cpu_gpu", "random_hotspots"]:
    show(f"results/stage5_inverse_q/{lay}/reconstruction.png")

## Stage 6 — inverse 3: hidden void in the TIM layer from top-surface temperatures

In [ ]:
!python scripts/stage6_defect.py --config configs/stage6_defect.yaml $EXTRA
show_md("results/stage6_defect/summary.md")
for f in ["k_profile", "top_surface", "stack_fields"]:
    show(f"results/stage6_defect/{f}.png")

## Stage 7 — ablation: vanilla vs Fourier features vs adaptive weights vs adaptive collocation

In [ ]:
!python scripts/stage7_ablation.py --config configs/stage7_ablation.yaml $EXTRA
show_md("results/stage7_ablation/ablation.md"); show("results/stage7_ablation/ablation.png")

## Stage 8a — sensor count (5/10/15/25) × placement (random vs QR-optimised)

In [ ]:
!python scripts/stage8_sensors.py --config configs/stage8_sensors.yaml $EXTRA
show_md("results/stage8_sensors/sensors.md")
show("results/stage8_sensors/sensor_count.png"); show("results/stage8_sensors/placements.png")

## Stage 8b — parametric PINN T(x, y, P) with silicon k(T), speed vs FV

In [ ]:
!python scripts/stage8_parametric.py --config configs/stage8_parametric.yaml $EXTRA
show_md("results/stage8_parametric/per_power.md"); show("results/stage8_parametric/peak_vs_power.png")

## Stage 9 — training GIF, README, download

In [ ]:
!python scripts/make_gif.py
!python scripts/make_report.py
show("results/figures/reconstruction.gif")

In [ ]:
# zip everything except checkpoints for download (Output tab or the link below)
!cd /kaggle/working && rm -f chip-pinn-results.zip && zip -qr chip-pinn-results.zip chip-pinn/results chip-pinn/README.md -x "*.pt" "*.pt.tmp"
from IPython.display import FileLink
FileLink("/kaggle/working/chip-pinn-results.zip") if os.path.exists("/kaggle/working/chip-pinn-results.zip") else print("zip not created")